# Build an MCP Agent

**Prerequisites:** all of `01`–`05`. From `02_agent_runtime`: `02_the_agent_loop.ipynb`

Every notebook in this chapter has made the same promise: `02_agent_runtime` and `03_tools` would not have to change. This one is the test, and it is short, because if the promise holds there is almost nothing left to do.

The claim being checked is narrow and worth stating exactly. Not *"an agent can use MCP tools"* — notebook 2 showed that. It is that the loop written four chapters ago, before any of this existed, runs a tool living in somebody else's process **without knowing it does**, and that every guarantee `03_tools` built still means what it meant.

One thing this notebook also does that the others deliberately avoided: it calls a real model. Everything up to now has been measurable without one, which is why the saved outputs have been worth reading. Here the interesting question — *does the refusal a server wrote actually change what the agent does next?* — is one only a live model can answer.

## Setup

In [1]:
import json
import sys
import time
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))
sys.path.append(str(Path.cwd().parent.parent / "02_agent_runtime"))
sys.path.append(str(Path.cwd().parent.parent / "03_tools"))

import psutil

from act import Act
from chat import Conversation, count_tokens
from observe import Observe
from registry import Registry
from runtime import Runtime
from think import Think

from remote import remote_tools
from session import Connection, Session

REGISTRY_SERVER = Path.cwd().parent / "registry_server.py"

# The shape this notebook arrives at, stated up front. The live session is kept on the
# conversation rather than inside the tools, and the last two sections are about why.
conversation = Conversation()
conversation.resources["mcp"] = Connection(lambda: Session(REGISTRY_SERVER)).open()

registry = Registry()
for t in remote_tools(conversation.resources["mcp"]):
    registry.add(t)

print("server:    ", conversation.resources["mcp"].server_name)
print("discovered:", registry.names)

server:     forge-registry
discovered: ['weather', 'pressure', 'save_note']


Three tools, none of which exist in this process. `weather` refuses politely when it has no station; `pressure` has a `KeyError` in it; `save_note` is wrapped in `verified()` and truncates what it saves.

## The host decides what the model sees

Notebook 1 named three roles and said the host was the one with no code in the SDK. This is it — and it is two lines.

In [2]:
everything = count_tokens(json.dumps([t.schema for t in registry]))

shown = registry.subset(names=["weather"])
chosen = count_tokens(json.dumps([t.schema for t in shown]))

print(f"connected to: {registry.names}  ({everything} tokens/turn)")
print(f"put in front of the model: {shown.names}  ({chosen} tokens/turn)")

connected to: ['weather', 'pressure', 'save_note']  (236 tokens/turn)
put in front of the model: ['weather']  (81 tokens/turn)


Connecting to a server and putting its tools in front of a model are two decisions, and `subset()` is where they come apart. The protocol will hand you everything a server offers; deciding what belongs in the prompt is a judgement it cannot make for you.

Here it is also keeping the notebook honest. `pressure` is a broken tool and `save_note` writes things — neither belongs in a demonstration about whether a refusal reaches the model, and leaving them in would let the run wander.

## The loop, unchanged

This is `03_tools` notebook 3's runtime, copied. The only difference is where the tools came from.

In [3]:
runtime = Runtime(
    loop=[
        Observe(event="What's the weather in Vestby, Norway?"),
        Think(tools=shown.tools, allow_fork=False),
        Act(functions=shown.invokers(timeout=10, max_tokens=400), schemas=None),
        Observe(),
    ],
    repeat_from=1,
    max_iterations=8,
    max_tokens=8000,
)

runtime.run(conversation)      # the conversation built above, connection and all

for m in conversation.messages:
    print(f"- {m['role']:9} {str(m.get('content'))[:96]!r}")

- user      "What's the weather in Vestby, Norway?"
- decision  '{"city":"Vestby, Norway"}'
- tool      'No weather station for Vestby, Norway. Try a larger nearby city.'
- decision  'No weather station for Vestby, Norway. Try a larger nearby city.'
- assistant 'No weather station for Vestby, Norway. Try a larger nearby city.'


Read that trace against what each chapter contributed, because every line of it is somebody's guarantee holding:

- the **decision** came from `Think`, which read a schema built by `Tool.schema` in `03_tools`, published over the wire by `expose()` in notebook 4, and turned back into a `Tool` by `remote.py` in notebook 2
- the **tool** message is a `ToolError` raised by a function in another process. `expose()` read `Result.for_the_model`, decided it was for the model, and put it in `content` rather than `_meta`
- `remote.py` read `forge/kind: refused` and raised `ToolError`, so the client's own `invoke()` classified it identically to a local refusal
- and the **assistant** message is the sentence `weather` wrote, reaching a human

That last line is worth sitting with, because it is not the outcome I expected when writing this cell. The tool said *try a larger nearby city*; the plausible next move was for the model to try Oslo. It did not. It passed the advice on and stopped. Not every time: an earlier run of this cell did try Oslo, got the same refusal, and was stopped by `max_iterations` — so which of the two you see is the model's call, run to run.

That is a defensible answer — the user asked about Vestby, and "there is no station there, try a larger city nearby" is true and actionable by the person who asked — but it is worth being precise that **the model chose to relay rather than to retry**, because the failure mode this chapter has been guarding against looks identical from one turn away. An agent that stalls holding advice it was never shown, and an agent that received the advice and decided the user should see it, produce the same-shaped transcript. Only the middle of the trace tells them apart, which is the argument for `Result.kind` being a thing you can inspect rather than a thing you infer from behaviour.

What the cell does settle — and what no amount of offline testing could have — is that the sentence survived. A string written by a function in another process, for a caller it will never meet, crossed a protocol, two `invoke()` calls and a summariser without being flattened into "the tool failed". Every notebook in this chapter was in service of that, and it is the one thing only a live model could have confirmed.

## Nothing changed in chapters 2 and 3

Worth checking rather than asserting.

In [4]:
import ast
import inspect

import act
import invoke as invoke_module
import registry as registry_module
import runtime as runtime_module
import think
import tools as tools_module

def imports(module):
    """Top-level package names this module imports. Not a text search -- a mention of `mcp`
    in a comment is not a dependency, and this cell is about dependencies."""
    found = set()
    for node in ast.walk(ast.parse(inspect.getsource(module))):
        if isinstance(node, ast.Import):
            found.update(a.name.split(".")[0] for a in node.names)
        elif isinstance(node, ast.ImportFrom) and node.module and node.level == 0:
            found.add(node.module.split(".")[0])
    return found

for mod in (act, runtime_module, think, tools_module, registry_module, invoke_module):
    deps = imports(mod)
    print(f"{mod.__name__ + '.py':14} imports mcp? {str('mcp' in deps):<6} ({len(deps)} imports)")

print()
print("what Act was actually handed:", type(shown.invokers()["weather"]).__name__,
      "-- an ordinary callable")

act.py         imports mcp? False  (6 imports)
runtime.py     imports mcp? False  (8 imports)
think.py       imports mcp? False  (6 imports)
tools.py       imports mcp? False  (3 imports)
registry.py    imports mcp? False  (4 imports)
invoke.py      imports mcp? False  (6 imports)

what Act was actually handed: function -- an ordinary callable


`Act` calls a function. The function happens to send JSON-RPC down a pipe to a second Python process, and nothing in `02_agent_runtime` has any way to find that out, or any reason to want to.

## Where the session lives

A tool built by `remote.py` is a closure, and what it closes over is the thing worth looking at.

In [5]:
print("the tool's free variables:", registry["weather"].function.__code__.co_freevars)
print("what 'session' actually is: ", type(conversation.resources["mcp"]).__name__)
print("kept on the conversation:   ", list(conversation.resources))

the tool's free variables: ('mcp_tool', 'session')
what 'session' actually is:  Connection
kept on the conversation:    ['mcp']


The closure has a variable called `session`, and what is sitting in it is a **`Connection`** — a box holding the real session rather than the session itself.

That distinction was the last thing this chapter needed. Handed a `Session` directly, a tool seals it in: it works, and when the server dies there is no way to give that function a new one — every tool would have to be rebuilt and the `Registry` rebuilt with them. Handed a box, the tool keeps the box forever and the contents can be swapped underneath it.

And because the box sits on `conversation.resources`, anything holding the conversation can reach it. Components already do — that is what `.run(conversation)` means — so a reconnect step, a health check or a failover component all become possible without any of them knowing how tools are built.

`Conversation.resources` is a change to `01_foundations/chat.py`, and it is there because **two chapters asked for it independently.** `04_memory` notebook 5 found that nothing on a conversation could hold a memory store, so `remember` had to close over one. This chapter found the same for a live session, where it is worse: a store that is merely awkward to reach is still working, while a dead session needs replacing by somebody.

## When the server dies

Now the two halves meet. Kill the server underneath the agent.

In [6]:
def server_pid():
    return next((c.pid for c in psutil.Process().children(recursive=True)
                 if "python" in c.name().lower()), None)

acting = Act(functions=shown.invokers(timeout=10), schemas=None)

def ask_about(city):
    c = Conversation()
    c.messages.append({"role": "decision", "type": "tool", "tool": "weather",
                       "content": json.dumps({"city": city})})
    acting.run(c)
    return c.pending["content"]

pid = server_pid()
print("killing the server:", pid)
psutil.Process(pid).kill()
time.sleep(1.0)

print("what the conversation gets:", ask_about("Oslo"))

killing the server: 600


what the conversation gets: weather could not be completed.


`weather could not be completed.` — and nothing else, which is correct.

The transport raised, because the pipe is gone. `remote.py` did not catch it, so it reached `invoke()`'s `except Exception` arm and became `broken`. `broken` is the one kind whose `for_the_model` is False, so `invoker()` replaced the message with an acknowledgement and moved the cause to `Result.value`. A dead server is not something a model can fix, and handing it `MCPError: Connection closed` invites it to work around a fault it cannot see.

Then something reaches the connection and replaces it — here this cell, in a real agent a component:

In [7]:
conversation.resources["mcp"].reconnect()

print("reconnects:    ", conversation.resources["mcp"].reconnects)
print("new server pid:", server_pid())
print()
print("the same tools, never rebuilt:", ask_about("Oslo"))

reconnects:     1
new server pid: 12808

the same tools, never rebuilt: 4 degrees celsius, clear skies in Oslo.


The `Registry` was built once, before that second server process existed. Nothing in it was touched, `Act` was constructed once, and the schemas the model sees never changed. A process died and was replaced underneath all of it.

## Policy: deciding when to do that

`reconnect()` is a mechanism and answers nothing about *when*. How many attempts, how long to wait, when to give up, and who gets told are all decisions — and in this repo decisions live in a component, because `02_agent_runtime` says a component is anything with `.run(conversation)`.

Which is why this needs no change to chapter 2 either. The extension point was built four chapters ago; `Reconnect` in [`repair.py`](../repair.py) just uses it, exactly as `04_memory`'s `Recall` does.

```python
Runtime(loop=[Observe(...), Reconnect(), Think(...), Act(...), Observe()], repeat_from=1)
```

Two decisions inside it are worth arguing with rather than reading past.

**It asks the server instead of reading the conversation.** By the time a failure reaches `pending` it is a string — `execute_tool_call` ends with `str(result)`, so `kind` is gone and "the connection died" is indistinguishable from "this tool has a bug". So it does not infer. It probes. Checking the world instead of trusting the report is `verified()`'s move from `03_tools`.

**A successful repair is not told to the model.** It goes in the run log, for the same reason `for_the_model` exists: the model cannot act on "we reconnected", and spending context on it invites it to reason about infrastructure it does not control. A repair that *fails* is the opposite — "the weather service is unreachable" changes what is worth attempting, so that one becomes a message.

Here are the three cases, without a model involved:

In [8]:
from repair import Reconnect

def outcome(label, connection, **policy):
    fix = Reconnect(**policy)
    holder = Conversation()
    holder.resources["mcp"] = connection
    message = fix.run(holder)
    told = "nothing" if message is None else message["content"][:56] + "..."
    print(f"{label:22} repairs={fix.repairs}  model told: {told}")
    for line in fix.log:
        print(f"{'':22} log: {line[:76]}")

# 1. a healthy server
outcome("healthy", conversation.resources["mcp"])

# 2. a server that died but can be restarted
psutil.Process(server_pid()).kill()
time.sleep(0.8)
outcome("died, restartable", conversation.resources["mcp"])

# 3. a server that will never start again
broken = conversation.resources["mcp"]
broken.factory = lambda: Session(Path.cwd().parent / "no_such_server.py", startup=6)
psutil.Process(server_pid()).kill()
time.sleep(0.8)
outcome("never coming back", broken, attempts=2, wait=0.1)

healthy                repairs=0  model told: nothing


died, restartable      repairs=1  model told: nothing
                       log: mcp: reconnected on attempt 1


never coming back      repairs=0  model told: The mcp service is unreachable and its tools cannot be u...
                       log: mcp: attempt 1 failed to start: RuntimeError: session failed to open: Except
                       log: mcp: attempt 2 failed to start: RuntimeError: session failed to open: Except
                       log: mcp: gave up after 2 attempts


The healthy case costs one `tools/list` round trip — about **2.5 ms** over stdio — and does nothing else. The middle case repairs silently. The last one gives up and hands the model a sentence it can act on.

Two details found while building it, both worth knowing:

**`ping` does not work.** The obvious liveness check is the protocol's own `ping`, and `Client.send_ping()` sends it — but a server built with this SDK's `MCPServer` answers `MCPError: Method not found`. A health check built on `ping` reports every healthy server as dead. `tools/list` is the cheapest call every MCP server must implement, so that is the probe.

**A repair budget is not optional.** Without one, a server that fails on startup gets `attempts` tries *per iteration*, forever, and the run spends its whole budget restarting a process that was never going to work. `budget=2` above is what stops the third and fourth turns from trying at all.

In [9]:
conversation.close()
print("conversation closed; child processes left:", server_pid())

conversation closed; child processes left: None


## Where this leaves the chapter

The promise held. Across six notebooks, the changes to earlier chapters were:

| | |
|---|---|
| `02_agent_runtime` | **none** |
| `03_tools` | one: `invoker()` now honours `for_the_model`, a flag it already computed and nobody read |
| `01_foundations` | one: `Conversation` gained `resources`, a place to keep live things |

Neither was an MCP requirement, and it is worth being clear about that.

`invoker()` was a latent bug: `for_the_model` was computed and read by nobody, which was survivable while `broken` meant "a function in this repo has a bug" and stopped being survivable once it meant "a server we do not control declined to explain itself".

`Conversation.resources` is a missing field, not a bug, and the reason it was added here rather than argued about is that **two chapters needed it from opposite directions** — `04_memory` for a store, this one for a session. The runtime was left alone on purpose: the first design had `Runtime.run()` close the resources when a run ended, which breaks the moment an agent does two runs on one conversation. Closing belongs to whoever opened the conversation, which is why `Conversation` is a context manager instead.

And the sentence notebook 1 spent its length earning still holds at the end: **MCP does not change what a tool is, it changes who owns the process it runs in.** Every difficulty in this chapter — one bool instead of six kinds, a schema more permissive than we would have written, verification that only works on the far side, an environment handed over at launch — is a consequence of ownership, not of protocol design.

What is deliberately unfinished:

- **Failover across servers.** `Reconnect` repairs *one* connection. Choosing a different server when the first is gone needs two connections and a rule for preferring one, and the rule is the hard part — a second server with the same tool names is not the same server, and a `Registry` built from one does not describe the other
- **Authentication.** Irrelevant for stdio, unavoidable for HTTP, and `08_guardrails`' problem
- **The environment a server starts with.** `05_mcp_transport` measured it and added `inherit=False`; deciding what a tool should be *allowed* to reach is a guardrails question, and the argument is already there waiting